In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Face_Cropping_Project"
)

DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_DIR = PROJECT_DIR / "outputs"

IMAGE_DIR = DATASET_DIR / "images"

FINAL_METADATA_PATH = (
    OUTPUT_DIR / "final_face_detections.csv"
)

FINAL_CROP_DIR = (
    OUTPUT_DIR / "final_face_crops"
)

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

print("Project directory:", PROJECT_DIR)
print("Image directory  :", IMAGE_DIR)
print("Final crop dir   :", FINAL_CROP_DIR)
print("Metadata file    :", FINAL_METADATA_PATH)

print("\nProject exists :", PROJECT_DIR.exists())
print("Images exist  :", IMAGE_DIR.exists())
print("Crops exist   :", FINAL_CROP_DIR.exists())
print("Metadata exists:", FINAL_METADATA_PATH.exists())

Project directory: /content/drive/MyDrive/Face_Cropping_Project
Image directory  : /content/drive/MyDrive/Face_Cropping_Project/dataset/images
Final crop dir   : /content/drive/MyDrive/Face_Cropping_Project/outputs/final_face_crops
Metadata file    : /content/drive/MyDrive/Face_Cropping_Project/outputs/final_face_detections.csv

Project exists : True
Images exist  : True
Crops exist   : True
Metadata exists: True


In [3]:
# Phase 3 — Step 1
# Final dataset integrity check

from pathlib import Path
import pandas as pd

FINAL_METADATA_PATH = OUTPUT_DIR / "final_face_detections.csv"
FINAL_CROP_DIR = OUTPUT_DIR / "final_face_crops"

final_df = pd.read_csv(FINAL_METADATA_PATH)

print("=" * 70)
print("PHASE 3 — FINAL DATASET INTEGRITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Metadata count
# ------------------------------------------------------------

print("\nMetadata records:", len(final_df))

# ------------------------------------------------------------
# 2. Check crop files
# ------------------------------------------------------------

existing_files = 0
missing_files = []

for crop_path in final_df["crop_path"]:
    path = Path(crop_path)

    if path.exists():
        existing_files += 1
    else:
        missing_files.append(str(path))

print("Existing crop files:", existing_files)
print("Missing crop files :", len(missing_files))

# ------------------------------------------------------------
# 3. Check unique source images
# ------------------------------------------------------------

unique_images = final_df["image_name"].nunique()

print("\nUnique source images:", unique_images)

# ------------------------------------------------------------
# 4. Check images with zero final crops
# ------------------------------------------------------------

all_input_images = {
    p.name
    for p in IMAGE_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in image_extensions
}

final_images = set(final_df["image_name"])

images_without_crops = sorted(
    all_input_images - final_images
)

print("Input images:", len(all_input_images))
print("Images without final crops:", len(images_without_crops))

if images_without_crops:
    print("\nImages without crops:")
    print(images_without_crops)

# ------------------------------------------------------------
# 5. Check duplicate metadata records
# ------------------------------------------------------------

duplicate_records = final_df.duplicated().sum()

print("\nDuplicate metadata records:", duplicate_records)

# ------------------------------------------------------------
# 6. Count physical image files
# ------------------------------------------------------------

physical_crop_files = [
    p for p in FINAL_CROP_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_extensions
]

print("Physical crop image files:", len(physical_crop_files))

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if (
    existing_files == len(final_df)
    and len(missing_files) == 0
    and len(images_without_crops) == 0
    and duplicate_records == 0
    and len(physical_crop_files) == len(final_df)
):
    print("FINAL DATASET INTEGRITY: PASSED")
else:
    print("FINAL DATASET INTEGRITY: REVIEW REQUIRED")

print("=" * 70)

PHASE 3 — FINAL DATASET INTEGRITY CHECK

Metadata records: 4601
Existing crop files: 4601
Missing crop files : 0

Unique source images: 2204
Input images: 2204
Images without final crops: 0

Duplicate metadata records: 0
Physical crop image files: 4601

FINAL DATASET INTEGRITY: PASSED


In [4]:
# Phase 3 — Step 2
# Final dataset statistics

import pandas as pd
from pathlib import Path

final_df = pd.read_csv(FINAL_METADATA_PATH)

# ------------------------------------------------------------
# Calculate face area
# ------------------------------------------------------------

final_df["face_area"] = (
    (final_df["x2"] - final_df["x1"]) *
    (final_df["y2"] - final_df["y1"])
)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

detections_per_image = (
    final_df.groupby("image_name")
    .size()
)

print("=" * 70)
print("PHASE 3 — FINAL DATASET STATISTICS")
print("=" * 70)

print("\n--- DATASET SIZE ---")
print("Total face crops       :", len(final_df))
print("Unique source images   :", final_df["image_name"].nunique())

print("\n--- FACES PER IMAGE ---")
print(detections_per_image.describe())

print("\nMaximum faces in image :", detections_per_image.max())

# ------------------------------------------------------------
# Confidence
# ------------------------------------------------------------

print("\n--- CONFIDENCE ---")
print(final_df["confidence"].describe())

# ------------------------------------------------------------
# Face area
# ------------------------------------------------------------

print("\n--- FACE AREA (pixels²) ---")
print(final_df["face_area"].describe())

# ------------------------------------------------------------
# Crop width
# ------------------------------------------------------------

print("\n--- CROP WIDTH (pixels) ---")
print(final_df["crop_width"].describe())

# ------------------------------------------------------------
# Crop height
# ------------------------------------------------------------

print("\n--- CROP HEIGHT (pixels) ---")
print(final_df["crop_height"].describe())

# ------------------------------------------------------------
# Very small crops
# ------------------------------------------------------------

print("\n--- SMALL CROPS ---")

for threshold in [20, 30, 50, 75, 100]:

    count = (
        (final_df["crop_width"] < threshold) |
        (final_df["crop_height"] < threshold)
    ).sum()

    print(
        f"Width OR height < {threshold:>3}px : {count}"
    )

# ------------------------------------------------------------
# Confidence ranges
# ------------------------------------------------------------

print("\n--- CONFIDENCE RANGES ---")

confidence_ranges = [
    ("0.25–0.40", 0.25, 0.40),
    ("0.40–0.60", 0.40, 0.60),
    ("0.60–0.80", 0.60, 0.80),
    ("0.80–0.90", 0.80, 0.90),
    ("0.90–1.00", 0.90, 1.01),
]

for label, low, high in confidence_ranges:

    count = (
        (final_df["confidence"] >= low) &
        (final_df["confidence"] < high)
    ).sum()

    percentage = count / len(final_df) * 100

    print(
        f"{label}: {count:4d} "
        f"({percentage:6.2f}%)"
    )

# ------------------------------------------------------------
# Faces-per-image distribution
# ------------------------------------------------------------

print("\n--- NUMBER OF FACES PER IMAGE ---")

for threshold in [1, 2, 5, 10, 20]:

    count = (
        detections_per_image >= threshold
    ).sum()

    print(
        f"Images with >= {threshold:2d} faces: {count}"
    )

print("\n" + "=" * 70)
print("STATISTICS COMPLETE")
print("=" * 70)

PHASE 3 — FINAL DATASET STATISTICS

--- DATASET SIZE ---
Total face crops       : 4601
Unique source images   : 2204

--- FACES PER IMAGE ---
count    2204.000000
mean        2.087568
std         2.559337
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        40.000000
dtype: float64

Maximum faces in image : 40

--- CONFIDENCE ---
count    4601.000000
mean        0.828113
std         0.115651
min         0.250058
25%         0.835648
50%         0.862435
75%         0.881699
max         0.956446
Name: confidence, dtype: float64

--- FACE AREA (pixels²) ---
count    4.601000e+03
mean     4.011337e+04
std      1.061124e+05
min      2.080000e+02
25%      4.070000e+03
50%      1.168500e+04
75%      3.398400e+04
max      3.284859e+06
Name: face_area, dtype: float64

--- CROP WIDTH (pixels) ---
count    4601.000000
mean      131.107151
std       119.039851
min         9.000000
25%        57.000000
50%        94.000000
75%       163.000000
max      147

In [5]:
# Phase 3 — Step 3
# Save final dataset summary

import json

summary = {
    "project": "Face Cropping System",

    "detector": {
        "model": "YOLO11n-face",
        "confidence_threshold": 0.25,
        "image_size": 1280,
        "minimum_face_area_pixels": 200,
        "nms": "YOLO default"
    },

    "dataset": {
        "input_images": 2204,
        "final_face_crops": 4601,
        "images_with_faces": 2204,
        "images_without_faces": 0
    },

    "processing": {
        "initial_detections_at_conf_025": 4978,
        "removed_small_detections": 377,
        "failed_images": 0,
        "missing_crop_files": 0,
        "duplicate_metadata_records": 0
    },

    "final_statistics": {
        "average_faces_per_image": round(
            detections_per_image.mean(), 4
        ),
        "median_faces_per_image": round(
            detections_per_image.median(), 4
        ),
        "maximum_faces_per_image": int(
            detections_per_image.max()
        ),

        "mean_confidence": round(
            final_df["confidence"].mean(), 4
        ),
        "median_confidence": round(
            final_df["confidence"].median(), 4
        ),

        "minimum_face_area": int(
            final_df["face_area"].min()
        ),
        "maximum_face_area": int(
            final_df["face_area"].max()
        ),

        "mean_crop_width": round(
            final_df["crop_width"].mean(), 2
        ),
        "mean_crop_height": round(
            final_df["crop_height"].mean(), 2
        )
    }
}

SUMMARY_PATH = OUTPUT_DIR / "final_dataset_summary.json"

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=4)

print("=" * 70)
print("FINAL DATASET SUMMARY SAVED")
print("=" * 70)

print("\nSummary file:")
print(SUMMARY_PATH)

print("\nFinal configuration:")
print(json.dumps(summary["detector"], indent=4))

print("\nFinal dataset:")
print(json.dumps(summary["dataset"], indent=4))

print("\nFinal statistics:")
print(json.dumps(summary["final_statistics"], indent=4))

FINAL DATASET SUMMARY SAVED

Summary file:
/content/drive/MyDrive/Face_Cropping_Project/outputs/final_dataset_summary.json

Final configuration:
{
    "model": "YOLO11n-face",
    "confidence_threshold": 0.25,
    "image_size": 1280,
    "minimum_face_area_pixels": 200,
    "nms": "YOLO default"
}

Final dataset:
{
    "input_images": 2204,
    "final_face_crops": 4601,
    "images_with_faces": 2204,
    "images_without_faces": 0
}

Final statistics:
{
    "average_faces_per_image": 2.0876,
    "median_faces_per_image": 1.0,
    "maximum_faces_per_image": 40,
    "mean_confidence": 0.8281,
    "median_confidence": 0.8624,
    "minimum_face_area": 208,
    "maximum_face_area": 3284859,
    "mean_crop_width": 131.11,
    "mean_crop_height": 169.52
}
